# ABSA Notebook for Colab

This notebook walks through the project in plain, step-by-step language.

We will:
1. load the raw XML files
2. turn them into simple tables
3. inspect the dataset
4. extract aspect terms
5. map each aspect to a category
6. predict sentiment

The goal here is clarity, so the notebook uses short comments and avoids extra notebook-only complexity where possible.

In [ ]:
from __future__ import annotations

import json
import os
import re
import shutil
import sys
import tempfile
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

# Matplotlib sometimes needs a writable temp folder in Colab.
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "absa_project_mpl"))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

IN_COLAB = "google.colab" in sys.modules
DEFAULT_DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/absa_project")

# In Colab we read the project from Drive.
# If we run locally, we fall back to the repo folder.
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = DEFAULT_DRIVE_PROJECT_DIR
    NOTEBOOK_ROOT = PROJECT_ROOT / "colab"
else:
    NOTEBOOK_ROOT = Path.cwd().resolve()
    if not (NOTEBOOK_ROOT / "one.ipynb").exists() and (NOTEBOOK_ROOT / "colab" / "one.ipynb").exists():
        NOTEBOOK_ROOT = NOTEBOOK_ROOT / "colab"
    PROJECT_ROOT = NOTEBOOK_ROOT.parent

RAW_DIR = NOTEBOOK_ROOT / "data" / "raw"
PROCESSED_DIR = NOTEBOOK_ROOT / "data" / "processed"
OUTPUT_DIR = NOTEBOOK_ROOT / "outputs"
EDA_DIR = OUTPUT_DIR / "eda"
MODELS_DIR = OUTPUT_DIR / "models"

SOURCE_RAW_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_XML = RAW_DIR / "Restaurants_Train_v2.xml"
TEST_XML = RAW_DIR / "Restaurants_Test_Gold.xml"

CATEGORY_ORDER = [
    "food",
    "service",
    "price",
    "ambience",
    "anecdotes/miscellaneous",
]
POLARITY_ORDER = ["positive", "negative", "neutral", "conflict"]

for folder in [RAW_DIR, PROCESSED_DIR, EDA_DIR, MODELS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Copy the raw XML files into the notebook workspace so the rest of the notebook
# can run from one predictable location.
if SOURCE_RAW_DIR.exists() and SOURCE_RAW_DIR != RAW_DIR:
    shutil.copytree(SOURCE_RAW_DIR, RAW_DIR, dirs_exist_ok=True)
    print(f"Copied raw data from: {SOURCE_RAW_DIR}")
else:
    print(f"Expecting raw XML files in: {RAW_DIR}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook workspace: {NOTEBOOK_ROOT}")

## 1. Data Preparation

This step reads the raw XML files and turns them into clean CSV tables.

We keep separate tables for sentences, aspect terms, and aspect categories because the later sections use them in different ways.

In [ ]:
# A small tokenizer is enough for this notebook.
TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z\-']+")


@dataclass(frozen=True)
class ParsedDataset:
    name: str
    sentences: pd.DataFrame
    aspects: pd.DataFrame
    categories: pd.DataFrame


def normalize_text(text: str) -> str:
    return " ".join((text or "").split())


def normalize_term(term: str) -> str:
    term = normalize_text(term).lower().strip()
    term = re.sub(r"[^a-z0-9\s\-']", " ", term)
    term = re.sub(r"\s+", " ", term)
    return term


def tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall((text or "").lower())


def parse_restaurant_xml(path: Path, split_name: str) -> ParsedDataset:
    root = ET.parse(path).getroot()

    sentence_rows: list[dict] = []
    aspect_rows: list[dict] = []
    category_rows: list[dict] = []

    for sentence in root.findall(".//sentence"):
        sentence_id = sentence.attrib["id"]
        text = normalize_text(sentence.findtext("text", default=""))

        aspect_terms_node = sentence.find("aspectTerms")
        aspect_categories_node = sentence.find("aspectCategories")

        aspect_terms = aspect_terms_node.findall("aspectTerm") if aspect_terms_node is not None else []
        aspect_categories = (
            aspect_categories_node.findall("aspectCategory")
            if aspect_categories_node is not None
            else []
        )

        # We save one sentence row with a few helpful counts for later analysis.
        sentence_rows.append(
            {
                "split": split_name,
                "sentence_id": sentence_id,
                "text": text,
                "token_count": len(tokenize(text)),
                "aspect_term_count": len(aspect_terms),
                "aspect_category_count": len(aspect_categories),
                "has_aspect_term": int(bool(aspect_terms)),
                "has_multiple_categories": int(len(aspect_categories) > 1),
            }
        )

        # Aspect terms and categories are stored separately because they are used by
        # different later tasks.
        for idx, aspect in enumerate(aspect_terms):
            term = aspect.attrib.get("term", "")
            aspect_rows.append(
                {
                    "split": split_name,
                    "sentence_id": sentence_id,
                    "aspect_id": f"{sentence_id}::term::{idx}",
                    "text": text,
                    "term": term,
                    "term_normalized": normalize_term(term),
                    "polarity": aspect.attrib.get("polarity", "").lower(),
                    "char_from": int(aspect.attrib.get("from", -1)),
                    "char_to": int(aspect.attrib.get("to", -1)),
                }
            )

        for idx, category in enumerate(aspect_categories):
            category_rows.append(
                {
                    "split": split_name,
                    "sentence_id": sentence_id,
                    "category_id": f"{sentence_id}::cat::{idx}",
                    "text": text,
                    "category": category.attrib.get("category", "").lower(),
                    "polarity": category.attrib.get("polarity", "").lower(),
                }
            )

    return ParsedDataset(
        name=split_name,
        sentences=pd.DataFrame(sentence_rows),
        aspects=pd.DataFrame(aspect_rows),
        categories=pd.DataFrame(category_rows),
    )


def export_dataset(dataset: ParsedDataset) -> None:
    dataset.sentences.to_csv(PROCESSED_DIR / f"{dataset.name}_sentences.csv", index=False)
    dataset.aspects.to_csv(PROCESSED_DIR / f"{dataset.name}_aspect_terms.csv", index=False)
    dataset.categories.to_csv(PROCESSED_DIR / f"{dataset.name}_aspect_categories.csv", index=False)


def build_dataset_summary(train: ParsedDataset, test: ParsedDataset) -> dict:
    def polarity_counts(df: pd.DataFrame) -> dict[str, int]:
        counts = df["polarity"].value_counts().to_dict()
        return {label: int(counts.get(label, 0)) for label in POLARITY_ORDER}

    def category_counts(df: pd.DataFrame) -> dict[str, int]:
        counts = df["category"].value_counts().to_dict()
        return {label: int(counts.get(label, 0)) for label in CATEGORY_ORDER}

    return {
        "train": {
            "sentence_count": int(train.sentences.shape[0]),
            "aspect_term_count": int(train.aspects.shape[0]),
            "aspect_category_count": int(train.categories.shape[0]),
            "aspect_term_polarities": polarity_counts(train.aspects),
            "aspect_category_distribution": category_counts(train.categories),
        },
        "test": {
            "sentence_count": int(test.sentences.shape[0]),
            "aspect_term_count": int(test.aspects.shape[0]),
            "aspect_category_count": int(test.categories.shape[0]),
            "aspect_term_polarities": polarity_counts(test.aspects),
            "aspect_category_distribution": category_counts(test.categories),
        },
        "notes": {
            "full_dataset_present": bool(TRAIN_XML.exists() and TEST_XML.exists()),
            "format": "SemEval 2014 restaurant XML with sentence text, aspect terms, and aspect categories.",
        },
    }


train = parse_restaurant_xml(TRAIN_XML, "train")
test = parse_restaurant_xml(TEST_XML, "test")

export_dataset(train)
export_dataset(test)

dataset_summary = build_dataset_summary(train, test)
with open(PROCESSED_DIR / "dataset_summary.json", "w", encoding="utf-8") as handle:
    json.dump(dataset_summary, handle, indent=2)

print("Prepared notebook copies in colab/data/processed.")
pd.DataFrame(
    [
        {"split": "train", **dataset_summary["train"]},
        {"split": "test", **dataset_summary["test"]},
    ]
)[["split", "sentence_count", "aspect_term_count", "aspect_category_count"]]

## 2. Quick Data Check

Here we do a quick pass over the dataset so the model results make more sense later.

Nothing fancy here: just a few counts, charts, and short summary numbers.

In [ ]:
def save_bar_chart(series: pd.Series, title: str, output_name: str, xlabel: str, ylabel: str) -> None:
    plt.figure(figsize=(10, 5))
    series.plot(kind="bar")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(EDA_DIR / output_name, dpi=180)
    plt.close()


def save_histogram(series: pd.Series, title: str, output_name: str, xlabel: str, bins: int = 20) -> None:
    plt.figure(figsize=(10, 5))
    plt.hist(series, bins=bins)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(EDA_DIR / output_name, dpi=180)
    plt.close()


# Reload the processed CSVs so this section can be run on its own.
train_sentences = pd.read_csv(PROCESSED_DIR / "train_sentences.csv")
test_sentences = pd.read_csv(PROCESSED_DIR / "test_sentences.csv")
train_aspects = pd.read_csv(PROCESSED_DIR / "train_aspect_terms.csv")
test_aspects = pd.read_csv(PROCESSED_DIR / "test_aspect_terms.csv")
train_categories = pd.read_csv(PROCESSED_DIR / "train_aspect_categories.csv")
test_categories = pd.read_csv(PROCESSED_DIR / "test_aspect_categories.csv")

dataset_overview = pd.DataFrame(
    [
        {
            "split": "train",
            "sentences": int(train_sentences.shape[0]),
            "aspect_terms": int(train_aspects.shape[0]),
            "aspect_categories": int(train_categories.shape[0]),
            "avg_tokens_per_sentence": round(train_sentences["token_count"].mean(), 2),
            "avg_aspects_per_sentence": round(train_sentences["aspect_term_count"].mean(), 2),
            "share_multi_category_sentences": round(train_sentences["has_multiple_categories"].mean(), 4),
        },
        {
            "split": "test",
            "sentences": int(test_sentences.shape[0]),
            "aspect_terms": int(test_aspects.shape[0]),
            "aspect_categories": int(test_categories.shape[0]),
            "avg_tokens_per_sentence": round(test_sentences["token_count"].mean(), 2),
            "avg_aspects_per_sentence": round(test_sentences["aspect_term_count"].mean(), 2),
            "share_multi_category_sentences": round(test_sentences["has_multiple_categories"].mean(), 4),
        },
    ]
)
dataset_overview.to_csv(EDA_DIR / "dataset_overview.csv", index=False)

term_polarities = (
    pd.concat([train_aspects.assign(split="train"), test_aspects.assign(split="test")])
    .groupby(["split", "polarity"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=POLARITY_ORDER, fill_value=0)
)
term_polarities.to_csv(EDA_DIR / "term_polarity_distribution.csv")

category_distribution = (
    pd.concat([train_categories.assign(split="train"), test_categories.assign(split="test")])
    .groupby(["split", "category"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=CATEGORY_ORDER, fill_value=0)
)
category_distribution.to_csv(EDA_DIR / "category_distribution.csv")

top_terms = (
    train_aspects["term_normalized"]
    .value_counts()
    .head(20)
    .rename_axis("term")
    .reset_index(name="count")
)
top_terms.to_csv(EDA_DIR / "top_train_aspect_terms.csv", index=False)

save_bar_chart(
    train_aspects["polarity"].value_counts().reindex(POLARITY_ORDER, fill_value=0),
    "Train aspect-term polarity distribution",
    "train_term_polarity_distribution.png",
    "Polarity",
    "Aspect terms",
)
save_bar_chart(
    train_categories["category"].value_counts().reindex(CATEGORY_ORDER, fill_value=0),
    "Train aspect-category distribution",
    "train_category_distribution.png",
    "Category",
    "Category annotations",
)
save_histogram(
    train_sentences["token_count"],
    "Train sentence length distribution",
    "train_sentence_length_histogram.png",
    "Tokens per sentence",
    bins=25,
)
save_histogram(
    train_sentences["aspect_term_count"],
    "Train aspect terms per sentence",
    "train_aspects_per_sentence_histogram.png",
    "Aspect terms per sentence",
    bins=10,
)
save_bar_chart(
    top_terms.set_index("term")["count"],
    "Top 20 train aspect terms",
    "top_train_aspect_terms.png",
    "Aspect term",
    "Count",
)

findings = {
    "positive_term_share_train": round((train_aspects["polarity"] == "positive").mean(), 4),
    "negative_term_share_train": round((train_aspects["polarity"] == "negative").mean(), 4),
    "conflict_term_share_train": round((train_aspects["polarity"] == "conflict").mean(), 4),
    "food_category_share_train": round((train_categories["category"] == "food").mean(), 4),
    "multi_category_sentence_share_train": round(train_sentences["has_multiple_categories"].mean(), 4),
    "median_tokens_per_sentence_train": int(train_sentences["token_count"].median()),
    "median_aspect_terms_per_sentence_train": int(train_sentences["aspect_term_count"].median()),
}
with open(EDA_DIR / "eda_findings.json", "w", encoding="utf-8") as handle:
    json.dump(findings, handle, indent=2)

dataset_overview

## 3. Aspect Extraction

This is a simple rule-based extractor.

Instead of training a heavy sequence model, we learn common aspect words from the training set and look for similar words in new sentences.

In [ ]:
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "but", "by", "for", "from", "had", "has",
    "have", "he", "her", "here", "him", "his", "i", "if", "in", "into", "is", "it", "its", "me",
    "my", "of", "on", "or", "our", "she", "so", "that", "the", "their", "them", "there", "they",
    "this", "to", "too", "us", "was", "we", "were", "with", "you", "your",
}

NON_ASPECT_WORDS = {
    "amazing", "awful", "bad", "better", "best", "delicious", "excellent", "friendly", "good",
    "great", "horrible", "love", "loved", "nice", "poor", "slow", "terrible", "wonderful",
}

MIN_SINGLE_TERM_COUNT = 2
MIN_MULTI_TERM_COUNT = 2
MIN_HEADWORD_COUNT = 3
MAX_MULTIWORD_LENGTH = 3


def build_term_resources(train_aspects: pd.DataFrame) -> tuple[set[str], set[str], set[str]]:
    term_counts = Counter(train_aspects["term_normalized"])
    headword_counts = Counter(train_aspects["term_normalized"].apply(lambda term: term.split()[-1]))

    # These small lexicons are the main memory of the extractor.
    single_word_terms = {
        term
        for term, count in term_counts.items()
        if count >= MIN_SINGLE_TERM_COUNT and len(term.split()) == 1
    }
    multi_word_terms = {
        term
        for term, count in term_counts.items()
        if count >= MIN_MULTI_TERM_COUNT and 1 < len(term.split()) <= MAX_MULTIWORD_LENGTH
    }
    frequent_headwords = {
        headword
        for headword, count in headword_counts.items()
        if count >= MIN_HEADWORD_COUNT and headword not in STOPWORDS
    }
    return single_word_terms, multi_word_terms, frequent_headwords


def extract_candidates(
    text: str,
    single_word_terms: set[str],
    multi_word_terms: set[str],
    frequent_headwords: set[str],
) -> list[tuple[str, str]]:
    tokens = tokenize(text)
    predictions: list[tuple[str, str]] = []
    seen: set[str] = set()

    # Try longer phrases first so we do not split something like "wine list" too early.
    for size in range(MAX_MULTIWORD_LENGTH, 1, -1):
        for start in range(0, len(tokens) - size + 1):
            phrase = " ".join(tokens[start:start + size])
            if phrase in multi_word_terms and phrase not in seen:
                predictions.append((phrase, "train_lexicon_multiword"))
                seen.add(phrase)

    for token in tokens:
        if token in single_word_terms and token not in seen:
            predictions.append((token, "train_lexicon_single"))
            seen.add(token)

    # If we still have nothing, allow a careful fallback to frequent headwords.
    for token in tokens:
        if token in seen:
            continue
        if token in STOPWORDS or token in NON_ASPECT_WORDS or len(token) < 3:
            continue
        if token in frequent_headwords:
            predictions.append((token, "noun_like_headword"))
            seen.add(token)

    return predictions


def evaluate_predictions(
    sentences: pd.DataFrame,
    gold_terms: pd.DataFrame,
    single_word_terms: set[str],
    multi_word_terms: set[str],
    frequent_headwords: set[str],
) -> tuple[pd.DataFrame, dict]:
    gold_lookup = (
        gold_terms.groupby("sentence_id")["term_normalized"]
        .apply(lambda values: sorted(set(values)))
        .to_dict()
    )

    rows: list[dict] = []
    true_positive = 0
    false_positive = 0
    false_negative = 0
    predicted_term_total = 0

    for row in sentences.itertuples(index=False):
        predictions = extract_candidates(row.text, single_word_terms, multi_word_terms, frequent_headwords)
        predicted_terms = sorted({term for term, _ in predictions})
        predicted_term_total += len(predicted_terms)
        gold_sentence_terms = set(gold_lookup.get(row.sentence_id, []))

        true_positive += len(set(predicted_terms).intersection(gold_sentence_terms))
        false_positive += len(set(predicted_terms) - gold_sentence_terms)
        false_negative += len(gold_sentence_terms - set(predicted_terms))

        if predictions:
            for term, source in predictions:
                rows.append(
                    {
                        "sentence_id": row.sentence_id,
                        "text": row.text,
                        "predicted_term_normalized": term,
                        "prediction_source": source,
                        "is_exact_gold_match": int(term in gold_sentence_terms),
                        "gold_terms_normalized": " | ".join(sorted(gold_sentence_terms)),
                    }
                )
        else:
            rows.append(
                {
                    "sentence_id": row.sentence_id,
                    "text": row.text,
                    "predicted_term_normalized": "",
                    "prediction_source": "no_prediction",
                    "is_exact_gold_match": 0,
                    "gold_terms_normalized": " | ".join(sorted(gold_sentence_terms)),
                }
            )

    precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) else 0.0
    recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    summary = {
        "precision": round(float(precision), 4),
        "recall": round(float(recall), 4),
        "f1": round(float(f1), 4),
        "sentences": int(len(sentences)),
        "gold_terms": int(len(gold_terms)),
        "predicted_terms": int(predicted_term_total),
        "average_predicted_terms_per_sentence": round(predicted_term_total / len(sentences), 4),
        "match_rule": "exact_normalized_match",
        "metric_note": (
            "Exact match is strict. A partly right span can still count as wrong if it does not match the gold term exactly."
        ),
        "method_note": (
            "This extractor is meant to be simple and explainable. It relies on common terms seen in training plus a small fallback rule."
        ),
        "limitation_note": (
            "This stage is measured on its own. The later sentiment step still uses gold aspect terms."
        ),
    }
    return pd.DataFrame(rows), summary


single_word_terms, multi_word_terms, frequent_headwords = build_term_resources(train_aspects)
extraction_predictions, extraction_summary = evaluate_predictions(
    test_sentences,
    test_aspects,
    single_word_terms,
    multi_word_terms,
    frequent_headwords,
)

extraction_predictions.to_csv(MODELS_DIR / "aspect_extraction_predictions.csv", index=False)
with open(MODELS_DIR / "aspect_extraction_summary.json", "w", encoding="utf-8") as handle:
    json.dump(extraction_summary, handle, indent=2)

extraction_summary

## 4. Aspect Category Mapping

Now we guess which category each aspect belongs to, such as `food` or `service`.

The logic stays practical: use learned patterns when we have enough examples, then fall back to simple keyword rules.

In [ ]:
SEED_KEYWORDS = {
    "food": {
        "food", "dish", "dishes", "meal", "meals", "menu", "pizza", "sushi", "wine", "drinks",
        "dessert", "desserts", "bread", "salad", "steak", "pasta", "soup", "burger", "sandwich",
        "appetizer", "appetizers", "fish", "portion", "portions", "taste", "flavor", "flavour",
        "fresh", "coffee", "beer", "cocktail", "roll", "rolls", "bagel", "bagels", "risotto",
    },
    "service": {
        "service", "staff", "waiter", "waiters", "waitress", "waitresses", "hostess", "manager",
        "delivery", "server", "servers", "reservation", "reservations", "check", "attentive",
    },
    "price": {
        "price", "prices", "priced", "expensive", "cheap", "affordable", "value", "worth", "cost",
        "bill", "bargain", "overpriced", "reasonable", "money",
    },
    "ambience": {
        "atmosphere", "ambience", "ambiance", "decor", "design", "music", "crowded", "space",
        "room", "rooms", "vibe", "romantic", "cozy", "garden", "chairs", "seats", "noisy", "quiet",
        "loud", "interior",
    },
    "anecdotes/miscellaneous": {
        "place", "restaurant", "spot", "experience", "experiences", "issue", "issues", "option",
        "options", "thing", "things", "deal", "deals",
    },
}

REGEX_PATTERNS = [
    ("price", re.compile(r"price|cost|bill|value|cheap|expensive|overpriced|affordable")),
    ("service", re.compile(r"staff|service|waiter|delivery|manager|reservation|server|hostess")),
    ("ambience", re.compile(r"atmosphere|ambience|ambiance|decor|music|seat|room|space|vibe")),
    ("anecdotes/miscellaneous", re.compile(r"place|restaurant|experience|issue|option|deal")),
]

SMOOTHING_ALPHA = 1.0
TERM_MIN_SUPPORT = 3
HEADWORD_MIN_SUPPORT = 5


def build_single_category_eval_frame(aspects: pd.DataFrame, categories: pd.DataFrame) -> pd.DataFrame:
    category_counts = categories.groupby("sentence_id")["category"].nunique().rename("category_count")
    single_category_sentences = category_counts[category_counts == 1].index
    single_category_map = (
        categories[categories["sentence_id"].isin(single_category_sentences)]
        .drop_duplicates(["sentence_id", "category"])
        .set_index("sentence_id")["category"]
    )
    eval_frame = aspects[aspects["sentence_id"].isin(single_category_sentences)].copy()
    eval_frame["gold_category"] = eval_frame["sentence_id"].map(single_category_map)
    return eval_frame


def seed_or_regex_match(term: str) -> tuple[str, str] | None:
    norm = normalize_term(term)
    tokens = set(norm.split())

    for category, keywords in SEED_KEYWORDS.items():
        if norm in keywords or tokens.intersection(keywords):
            return category, "seed_keyword"

    for category, pattern in REGEX_PATTERNS:
        if pattern.search(norm):
            return category, f"regex_{category.replace('/', '_')}"

    return None


def baseline_rule_predict(term: str) -> tuple[str, str]:
    matched = seed_or_regex_match(term)
    if matched is not None:
        return matched

    norm = normalize_term(term)
    tokens = set(norm.split())
    if any(token.endswith(("ing", "ed")) for token in tokens):
        return "service", "verbish_fallback"

    return "food", "default_food_fallback"


class HybridAspectMapper:
    def __init__(self, smoothing_alpha: float = SMOOTHING_ALPHA) -> None:
        self.smoothing_alpha = smoothing_alpha
        self.term_category_scores: dict[str, dict[str, float]] = {}
        self.headword_scores: dict[str, dict[str, float]] = {}
        self.term_support: dict[str, int] = {}
        self.headword_support: dict[str, int] = {}
        self.majority_category = "food"

    @staticmethod
    def headword(term: str) -> str:
        pieces = normalize_term(term).split()
        return pieces[-1] if pieces else ""

    def fit(self, train_eval_frame: pd.DataFrame) -> None:
        term_counts: dict[str, Counter] = defaultdict(Counter)
        headword_counts: dict[str, Counter] = defaultdict(Counter)

        # Learn how often each term and headword points to each category.
        for row in train_eval_frame.itertuples(index=False):
            term_counts[row.term_normalized][row.gold_category] += 1
            headword = self.headword(row.term_normalized)
            if headword:
                headword_counts[headword][row.gold_category] += 1

        self.term_support = {key: int(sum(counter.values())) for key, counter in term_counts.items()}
        self.headword_support = {key: int(sum(counter.values())) for key, counter in headword_counts.items()}
        self.term_category_scores = self._to_scores(term_counts)
        self.headword_scores = self._to_scores(headword_counts)
        self.majority_category = train_eval_frame["gold_category"].value_counts().idxmax()

    def _to_scores(self, counts: dict[str, Counter]) -> dict[str, dict[str, float]]:
        scores: dict[str, dict[str, float]] = {}
        category_count = len(CATEGORY_ORDER)
        alpha = self.smoothing_alpha

        for key, counter in counts.items():
            scores[key] = {}
            key_total = sum(counter.values())
            smoothed_key_total = key_total + alpha * category_count

            for category in CATEGORY_ORDER:
                probability = (counter.get(category, 0) + alpha) / smoothed_key_total
                scores[key][category] = round(float(probability), 6)

        return scores

    def _predict_from_learned(
        self,
        key: str,
        score_lookup: dict[str, dict[str, float]],
        support_lookup: dict[str, int],
        minimum_support: int,
        source_name: str,
    ) -> tuple[str, str] | None:
        if not key:
            return None
        if support_lookup.get(key, 0) < minimum_support:
            return None
        if key not in score_lookup:
            return None
        score_map = score_lookup[key]
        return max(score_map, key=score_map.get), source_name

    def predict(self, term: str) -> tuple[str, str]:
        norm = normalize_term(term)

        # Prefer learned evidence first, then fall back to simple rules.
        learned_prediction = self._predict_from_learned(
            norm,
            self.term_category_scores,
            self.term_support,
            TERM_MIN_SUPPORT,
            "term_support_score",
        )
        if learned_prediction is not None:
            return learned_prediction

        head = self.headword(norm)
        learned_headword_prediction = self._predict_from_learned(
            head,
            self.headword_scores,
            self.headword_support,
            HEADWORD_MIN_SUPPORT,
            "headword_support_score",
        )
        if learned_headword_prediction is not None:
            return learned_headword_prediction

        matched = seed_or_regex_match(term)
        if matched is not None:
            return matched

        return self.majority_category, "low_confidence_majority_fallback"


def evaluate_method(eval_frame: pd.DataFrame, predictor) -> tuple[pd.DataFrame, dict]:
    frame = eval_frame.copy()
    predictions = frame["term"].apply(predictor)
    frame["predicted_category"] = [item[0] for item in predictions]
    frame["prediction_source"] = [item[1] for item in predictions]
    frame["is_low_confidence"] = frame["prediction_source"].eq("low_confidence_majority_fallback")

    accuracy = accuracy_score(frame["gold_category"], frame["predicted_category"])
    report = classification_report(
        frame["gold_category"],
        frame["predicted_category"],
        labels=CATEGORY_ORDER,
        output_dict=True,
        zero_division=0,
    )
    macro_f1 = report["macro avg"]["f1-score"]

    metrics = {
        "rows": int(frame.shape[0]),
        "accuracy": round(float(accuracy), 4),
        "macro_f1": round(float(macro_f1), 4),
        "coverage": 1.0,
        "low_confidence_predictions": int(frame["is_low_confidence"].sum()),
    }
    return frame, metrics


def build_source_breakdown(frame: pd.DataFrame) -> pd.DataFrame:
    breakdown = (
        frame.groupby("prediction_source")
        .agg(
            rows=("prediction_source", "size"),
            accuracy=("predicted_category", lambda col: float((col == frame.loc[col.index, "gold_category"]).mean())),
            low_confidence=("is_low_confidence", "sum"),
        )
        .reset_index()
        .sort_values(["rows", "prediction_source"], ascending=[False, True])
    )
    breakdown["accuracy"] = breakdown["accuracy"].round(4)
    return breakdown


train_eval = build_single_category_eval_frame(train_aspects, train_categories)
test_eval = build_single_category_eval_frame(test_aspects, test_categories)

baseline_predictions, baseline_metrics = evaluate_method(test_eval, baseline_rule_predict)

mapper = HybridAspectMapper()
mapper.fit(train_eval)
hybrid_predictions, hybrid_metrics = evaluate_method(test_eval, mapper.predict)

train_eval.to_csv(MODELS_DIR / "aspect_generation_train_eval_frame.csv", index=False)
test_eval.to_csv(MODELS_DIR / "aspect_generation_test_eval_frame.csv", index=False)
baseline_predictions.to_csv(MODELS_DIR / "aspect_generation_baseline_predictions.csv", index=False)
hybrid_predictions.to_csv(MODELS_DIR / "aspect_generation_hybrid_predictions.csv", index=False)

source_breakdown = build_source_breakdown(hybrid_predictions)
source_breakdown.to_csv(MODELS_DIR / "aspect_generation_source_breakdown.csv", index=False)

aspect_generation_summary = {
    "evaluation_note": (
        "We only score sentences that have one category label. Multi-category sentences do not tell us which aspect belongs to which category."
    ),
    "metric_note": "Macro-F1 is the main metric because some categories appear much more often than others.",
    "method_note": (
        "The hybrid mapper first uses learned term patterns, then headword patterns, then simple keyword rules, and finally a low-confidence fallback."
    ),
    "limitation_note": (
        "Category mapping is checked separately from raw extraction. Low-confidence fallback cases are tracked in the saved outputs."
    ),
    "train_eval_rows": int(train_eval.shape[0]),
    "test_eval_rows": int(test_eval.shape[0]),
    "smoothing_alpha": SMOOTHING_ALPHA,
    "term_min_support": TERM_MIN_SUPPORT,
    "headword_min_support": HEADWORD_MIN_SUPPORT,
    "majority_category_fallback": mapper.majority_category,
    "baseline": baseline_metrics,
    "hybrid": hybrid_metrics,
    "hybrid_source_counts": {
        row["prediction_source"]: int(row["rows"])
        for row in source_breakdown.to_dict(orient="records")
    },
    "improvement_accuracy": round(hybrid_metrics["accuracy"] - baseline_metrics["accuracy"], 4),
    "improvement_macro_f1": round(hybrid_metrics["macro_f1"] - baseline_metrics["macro_f1"], 4),
}
with open(MODELS_DIR / "aspect_generation_summary.json", "w", encoding="utf-8") as handle:
    json.dump(aspect_generation_summary, handle, indent=2)

term_lexicon_rows = []
for term, score_map in sorted(mapper.term_category_scores.items()):
    best_category = max(score_map, key=score_map.get)
    term_lexicon_rows.append(
        {
            "term_normalized": term,
            "support": mapper.term_support.get(term, 0),
            "predicted_category": best_category,
            "best_score": score_map[best_category],
        }
    )
pd.DataFrame(term_lexicon_rows).to_csv(MODELS_DIR / "learned_term_category_lexicon.csv", index=False)

aspect_generation_summary

## 5. Sentiment Modelling

Last, we predict whether each aspect is talked about in a positive, negative, neutral, or mixed way.

To keep this notebook easier to follow, we use one clear TF-IDF + Logistic Regression model instead of comparing several similar models.

In [ ]:
def mark_aspect(sentence: str, term: str) -> str:
    sentence_lower = sentence.lower()
    term_lower = term.lower()
    start = sentence_lower.find(term_lower)
    if start == -1:
        return f"[ASPECT] {term} [/ASPECT] || {sentence}"
    end = start + len(term)
    return sentence[:start] + "[ASPECT] " + sentence[start:end] + " [/ASPECT]" + sentence[end:]


def build_features(frame: pd.DataFrame) -> pd.Series:
    # Mark the aspect inside the sentence so the model knows what opinion to judge.
    return frame.apply(
        lambda row: f"term={row['term_normalized']} || context={mark_aspect(row['text'], row['term'])}",
        axis=1,
    )


def majority_baseline(y_train: pd.Series, y_test: pd.Series) -> dict:
    majority_label = y_train.value_counts().idxmax()
    predictions = pd.Series([majority_label] * len(y_test), index=y_test.index)
    return {
        "model_name": "majority_baseline",
        "label": majority_label,
        "accuracy": round(float(accuracy_score(y_test, predictions)), 4),
        "macro_f1": round(float(f1_score(y_test, predictions, average="macro", zero_division=0)), 4),
        "rows": int(len(y_test)),
    }


def train_simple_sentiment_model(X_train_text: pd.Series, y_train: pd.Series):
    # Keep the notebook model simple and easy to explain.
    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000,
        sublinear_tf=True,
    )
    X_train_vec = vectorizer.fit_transform(X_train_text)

    model = LogisticRegression(
        max_iter=4000,
        class_weight="balanced",
        solver="lbfgs",
        random_state=42,
    )
    model.fit(X_train_vec, y_train)
    return vectorizer, model


X_train = build_features(train_aspects)
y_train = train_aspects["polarity"]
X_test = build_features(test_aspects)
y_test = test_aspects["polarity"]

baseline = majority_baseline(y_train, y_test)

vectorizer, model = train_simple_sentiment_model(X_train, y_train)
X_test_vec = vectorizer.transform(X_test)
test_pred = model.predict(X_test_vec)

model_metrics = {
    "model_name": "logreg_word_tfidf",
    "accuracy": round(float(accuracy_score(y_test, test_pred)), 4),
    "macro_f1": round(float(f1_score(y_test, test_pred, average="macro", zero_division=0)), 4),
    "rows": int(len(y_test)),
}

comparison_df = pd.DataFrame([baseline, model_metrics])
comparison_df.to_csv(MODELS_DIR / "sentiment_model_comparison.csv", index=False)

report = classification_report(
    y_test,
    test_pred,
    labels=POLARITY_ORDER,
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(report).transpose().to_csv(MODELS_DIR / "sentiment_classification_report.csv")

confusion_df = pd.DataFrame(
    confusion_matrix(y_test, test_pred, labels=POLARITY_ORDER),
    index=[f"gold_{label}" for label in POLARITY_ORDER],
    columns=[f"pred_{label}" for label in POLARITY_ORDER],
)
confusion_df.to_csv(MODELS_DIR / "sentiment_confusion_matrix.csv")

predictions = test_aspects[["sentence_id", "term", "term_normalized", "polarity"]].copy()
predictions["selected_model"] = model_metrics["model_name"]
predictions["predicted_polarity"] = test_pred
predictions.to_csv(MODELS_DIR / "sentiment_test_predictions.csv", index=False)

sentiment_summary = {
    "majority_baseline": baseline,
    "selected_model": model_metrics["model_name"],
    "selected_model_results": model_metrics,
    "all_model_results": comparison_df.to_dict(orient="records"),
    "metric_note": "Macro-F1 is still the most useful score here because the sentiment labels are imbalanced.",
    "model_note": (
        "This notebook keeps the sentiment step simple: TF-IDF features plus Logistic Regression on the marked sentence context."
    ),
    "limitation_note": (
        "This stage still uses the gold aspect terms, so it measures sentiment prediction and not the full end-to-end pipeline."
    ),
}
with open(MODELS_DIR / "sentiment_summary.json", "w", encoding="utf-8") as handle:
    json.dump(sentiment_summary, handle, indent=2)

sentiment_summary